# AG News Text Classification — Task 1

**Model architectures · training pipeline · evaluation · hosted PostgreSQL persistence**

This notebook is designed for **Kaggle Notebooks with a GPU**. It intentionally does **not** contain precomputed or fabricated results: training and evaluation run only when you execute the cells.

**Scope:** the model/training/evaluation portion of Task 1. It includes a small amount of dataset verification and transparent text normalization so the pipeline is independently runnable; coordinate final dataset-preprocessing descriptions with your teammate.

**Evaluation design:** split the official AG News training split into a stratified train/validation partition. Compare all 9 traditional representation–algorithm combinations plus LSTM and BiLSTM on validation; choose models using validation Macro-F1 only. Refit the validation-selected traditional model and both recurrent architectures on the full official training split, then evaluate those finalists on the official test split. This keeps the official test set out of model selection while producing robust final test reports.

> Run the notebook from top to bottom. It is intentionally unexecuted here, as requested.

## 1. Rubric coverage

| Task 1 requirement | Where it is addressed |
|---|---|
| Dataset, labels, class balance and preprocessing | Dataset loading, distribution tables, text-normalization notes |
| At least 3 text representations | Count/BoW, word-level TF-IDF, character-level TF-IDF |
| At least 3 traditional algorithms and 6+ valid combinations | Multinomial Naive Bayes, Logistic Regression, Linear SVM; full 3 × 3 = 9 grid |
| LSTM and BiLSTM classifiers | PyTorch implementations with train/validation loops and early stopping |
| Accuracy, Macro-F1, and training-time comparison | Validation comparison table plus finalist test table and timings |
| Confusion matrices for best traditional model and LSTM | Final test confusion matrices for the validation-selected traditional model and LSTM; BiLSTM is reported too |
| Representative prediction errors | Per-class report, confusion-pair counts, high-score mistakes, exportable error table |
| Reproducibility and result artifacts | Fixed seeds, recorded hyperparameters, checkpoints, CSV/JSON, plots |
| Online PostgreSQL persistence | Optional hosted PostgreSQL connection; schema creation, dataset storage, run metrics, epoch history, and predictions |

### Important methodological note

Hyperparameters are declared in the configuration cell. The validation scores choose the best traditional pipeline and the best epoch for each recurrent architecture. The held-out test set is only used after that selection. Do not change hyperparameters based on test-set outcomes; if you do, treat that test set as a development set and create a new holdout for a final estimate.

## 2. Dataset and project references

- **Dataset:** [fancyzhx/ag_news on Hugging Face](https://huggingface.co/datasets/fancyzhx/ag_news)
- The dataset viewer currently reports **120,000 training examples** and **7,600 test examples**, with four labels: `World`, `Sports`, `Business`, `Sci/Tech`. The notebook validates counts and label names from the loaded artifact rather than silently assuming them.
- Original reference: Xiang Zhang, Junbo Jake Zhao, and Yann LeCun, [“Character-level Convolutional Networks for Text Classification”](https://papers.nips.cc/paper_files/paper/2015/hash/250cf8b51c773f3f8dc8b4be867a9a02-Abstract.html), NeurIPS 2015.
- The Hugging Face dataset page currently displays its license as **unknown**. Check the dataset card and course policy before redistributing raw data or publishing artifacts that include article text.

**Label IDs used by the source dataset:** `0 = World`, `1 = Sports`, `2 = Business`, `3 = Sci/Tech`. The code prints the actual feature names and checks the mapping after loading.

## 3. Kaggle setup and secure PostgreSQL configuration

1. Create a hosted PostgreSQL database with a provider such as [Neon](https://neon.tech/) or [Supabase](https://supabase.com/), and copy its **connection URI**. Use the provider's SSL-enabled URI.
2. In Kaggle, open **Add-ons → Secrets**, add a secret named `POSTGRES_URL`, and paste the URI as its value. Grant the notebook access to that secret. Alternatively, set `POSTGRES_URL` as an environment variable in a trusted environment.
3. Never paste the URI into a notebook cell or commit it to a public repository. The notebook does not print the secret.
4. If the secret is absent, local training/export still works and database persistence is skipped with a warning. For this project requirement, configure the secret before the actual run.

The notebook creates its own tables in the connected database. A shared/free database may have storage and connection limits; avoid uploading the same project dataset to multiple databases unnecessarily.

In [ ]:
# Kaggle already includes PyTorch, NumPy, pandas, matplotlib and usually scikit-learn.
# Install only the extra/compatible dependencies needed by this notebook.
%pip -q install "datasets>=3.0,<5" "psycopg2-binary>=2.9" "scikit-learn>=1.3" "joblib>=1.3"

In [ ]:
import os
import re
import html
import json
import time
import uuid
import math
import random
import hashlib
import warnings
import platform
from collections import Counter
from copy import deepcopy
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.exceptions import ConvergenceWarning
from scipy.special import softmax
import joblib

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from datasets import load_dataset

try:
    import psycopg2
    from psycopg2.extras import execute_values, Json
except ImportError:
    psycopg2 = None
    execute_values = None
    Json = None

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ----------------------- Reproducible experiment configuration -----------------------
SEED = 42
DATASET_ID = "fancyzhx/ag_news"
LABEL_NAMES_FALLBACK = ["World", "Sports", "Business", "Sci/Tech"]
NUM_CLASSES = 4
VALIDATION_SIZE = 0.10

# Traditional sparse text features. Lower MAX_FEATURES if your Kaggle session is RAM-limited.
BOW_MAX_FEATURES = 100_000
WORD_TFIDF_MAX_FEATURES = 120_000
CHAR_TFIDF_MAX_FEATURES = 150_000

# Recurrent model settings
MAX_VOCAB_SIZE = 50_000          # includes PAD and UNK tokens
MIN_TOKEN_FREQUENCY = 2
MAX_SEQUENCE_LENGTH = 128
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
LSTM_LAYERS = 2
DROPOUT = 0.35
NN_BATCH_SIZE = 256
NN_EPOCHS = 6
NN_PATIENCE = 2
NN_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_MIXED_PRECISION = True       # auto-disabled on CPU
NUM_WORKERS = 0                  # robust default for hosted notebook kernels

OUTPUT_DIR = Path("/kaggle/working/ag_news_task1_outputs")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("./ag_news_task1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(USE_MIXED_PRECISION and DEVICE.type == "cuda")


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Improves repeatability; exact determinism can still vary by CUDA/library versions.
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

CONFIG = {
    "seed": SEED,
    "dataset_id": DATASET_ID,
    "validation_size": VALIDATION_SIZE,
    "bow_max_features": BOW_MAX_FEATURES,
    "word_tfidf_max_features": WORD_TFIDF_MAX_FEATURES,
    "char_tfidf_max_features": CHAR_TFIDF_MAX_FEATURES,
    "max_vocab_size": MAX_VOCAB_SIZE,
    "min_token_frequency": MIN_TOKEN_FREQUENCY,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "lstm_layers": LSTM_LAYERS,
    "dropout": DROPOUT,
    "nn_batch_size": NN_BATCH_SIZE,
    "nn_epochs": NN_EPOCHS,
    "nn_patience": NN_PATIENCE,
    "nn_learning_rate": NN_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "device": str(DEVICE),
    "mixed_precision": USE_AMP,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
}
print(json.dumps(CONFIG, indent=2))

## 4. Hosted PostgreSQL: schema and safe connection

The database stores raw dataset examples keyed by source split/index, experiment metadata, validation/test metrics, confusion matrices/reports, epoch histories, and model predictions. Model binaries are saved as notebook artifacts rather than placed inside PostgreSQL rows.

All statements use parameterized SQL. Dataset uploads are batched and idempotent (`ON CONFLICT DO NOTHING`).

In [ ]:
def read_postgres_url() -> str | None:
    """Read a connection URL from an environment variable or Kaggle Secret without printing it."""
    value = os.environ.get("POSTGRES_URL")
    if value:
        return value.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("POSTGRES_URL")
        if value:
            return value.strip()
    except Exception:
        # Expected outside Kaggle or when the named secret has not been added/granted.
        pass
    return None

POSTGRES_URL = read_postgres_url()
DB_ENABLED = bool(POSTGRES_URL and psycopg2 is not None)
if DB_ENABLED:
    print("PostgreSQL persistence enabled. Connection string is hidden.")
else:
    print("WARNING: PostgreSQL persistence is disabled. Add the Kaggle secret POSTGRES_URL to persist data/results online.")

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS dataset_examples (
    dataset_name TEXT NOT NULL,
    split TEXT NOT NULL CHECK (split IN ('train', 'test')),
    source_index INTEGER NOT NULL,
    text TEXT NOT NULL,
    label SMALLINT NOT NULL CHECK (label BETWEEN 0 AND 3),
    label_name TEXT NOT NULL,
    text_sha256 CHAR(64) NOT NULL,
    inserted_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (dataset_name, split, source_index)
);
CREATE INDEX IF NOT EXISTS idx_dataset_examples_label
    ON dataset_examples(dataset_name, split, label);

CREATE TABLE IF NOT EXISTS experiment_runs (
    run_id UUID PRIMARY KEY,
    dataset_name TEXT NOT NULL,
    seed INTEGER NOT NULL,
    device TEXT NOT NULL,
    status TEXT NOT NULL,
    config JSONB NOT NULL,
    started_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    completed_at TIMESTAMPTZ,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS model_metrics (
    run_id UUID NOT NULL REFERENCES experiment_runs(run_id) ON DELETE CASCADE,
    evaluation_stage TEXT NOT NULL,
    model_key TEXT NOT NULL,
    model_family TEXT NOT NULL,
    representation TEXT,
    accuracy DOUBLE PRECISION NOT NULL,
    macro_f1 DOUBLE PRECISION NOT NULL,
    weighted_f1 DOUBLE PRECISION NOT NULL,
    macro_precision DOUBLE PRECISION NOT NULL,
    macro_recall DOUBLE PRECISION NOT NULL,
    training_seconds DOUBLE PRECISION,
    vectorizer_fit_seconds DOUBLE PRECISION,
    best_epoch INTEGER,
    confusion_matrix JSONB NOT NULL,
    classification_report JSONB NOT NULL,
    model_config JSONB NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (run_id, evaluation_stage, model_key)
);

CREATE TABLE IF NOT EXISTS training_history (
    run_id UUID NOT NULL REFERENCES experiment_runs(run_id) ON DELETE CASCADE,
    model_key TEXT NOT NULL,
    training_stage TEXT NOT NULL,
    epoch INTEGER NOT NULL,
    train_loss DOUBLE PRECISION,
    validation_loss DOUBLE PRECISION,
    validation_accuracy DOUBLE PRECISION,
    validation_macro_f1 DOUBLE PRECISION,
    learning_rate DOUBLE PRECISION,
    PRIMARY KEY (run_id, model_key, training_stage, epoch)
);

CREATE TABLE IF NOT EXISTS model_predictions (
    run_id UUID NOT NULL,
    evaluation_stage TEXT NOT NULL,
    model_key TEXT NOT NULL,
    dataset_name TEXT NOT NULL,
    dataset_split TEXT NOT NULL CHECK (dataset_split IN ('train', 'test')),
    source_index INTEGER NOT NULL,
    true_label SMALLINT NOT NULL,
    predicted_label SMALLINT NOT NULL,
    score_confidence DOUBLE PRECISION,
    is_correct BOOLEAN NOT NULL,
    PRIMARY KEY (run_id, evaluation_stage, model_key, dataset_name, dataset_split, source_index),
    FOREIGN KEY (run_id, evaluation_stage, model_key)
        REFERENCES model_metrics(run_id, evaluation_stage, model_key) ON DELETE CASCADE,
    FOREIGN KEY (dataset_name, dataset_split, source_index)
        REFERENCES dataset_examples(dataset_name, split, source_index) ON DELETE CASCADE
);
CREATE INDEX IF NOT EXISTS idx_model_predictions_errors
    ON model_predictions(run_id, evaluation_stage, model_key, is_correct);
"""


def db_connect():
    if not DB_ENABLED:
        raise RuntimeError("PostgreSQL is not configured. Set POSTGRES_URL first.")
    # Hosted database URIs should use the provider's SSL-enabled connection string.
    return psycopg2.connect(POSTGRES_URL, sslmode="require", connect_timeout=15)


def initialize_database() -> None:
    if not DB_ENABLED:
        return
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(SCHEMA_SQL)
    print("PostgreSQL schema is ready.")

initialize_database()

## 5. Load AG News and persist the source examples

The database copy contains the **original dataset text and labels**. The normalized model input is kept as a separate in-memory column and is not used to overwrite source data.

In [ ]:
dataset = load_dataset(DATASET_ID)

train_raw = dataset["train"].to_pandas().reset_index(drop=True)
test_raw = dataset["test"].to_pandas().reset_index(drop=True)
train_raw["source_index"] = np.arange(len(train_raw), dtype=np.int32)
test_raw["source_index"] = np.arange(len(test_raw), dtype=np.int32)

label_feature = dataset["train"].features["label"]
LABEL_NAMES = list(getattr(label_feature, "names", LABEL_NAMES_FALLBACK))
LABEL_IDS = list(range(len(LABEL_NAMES)))
assert len(LABEL_NAMES) == NUM_CLASSES, f"Expected 4 categories, found {LABEL_NAMES}"
assert set(train_raw["label"].unique()).issubset(set(LABEL_IDS))
assert set(test_raw["label"].unique()).issubset(set(LABEL_IDS))

print("Dataset:", DATASET_ID)
print("Train shape:", train_raw.shape, "| Test shape:", test_raw.shape)
print("Label mapping:", dict(enumerate(LABEL_NAMES)))
print("Dataset features:", dataset["train"].features)


def persist_dataset_split(split_name: str, frame: pd.DataFrame, batch_size: int = 2000) -> None:
    if not DB_ENABLED:
        return
    insert_sql = """
        INSERT INTO dataset_examples
            (dataset_name, split, source_index, text, label, label_name, text_sha256)
        VALUES %s
        ON CONFLICT (dataset_name, split, source_index) DO NOTHING
    """
    texts = frame["text"].astype(str).tolist()
    labels = frame["label"].astype(int).tolist()
    label_names = [LABEL_NAMES[label] for label in labels]
    indices = frame["source_index"].astype(int).tolist()
    with db_connect() as conn:
        with conn.cursor() as cur:
            for start in range(0, len(frame), batch_size):
                stop = min(start + batch_size, len(frame))
                rows = []
                for i in range(start, stop):
                    raw_text = texts[i]
                    digest = hashlib.sha256(raw_text.encode("utf-8")).hexdigest()
                    rows.append((DATASET_ID, split_name, indices[i], raw_text, labels[i], label_names[i], digest))
                execute_values(cur, insert_sql, rows, page_size=500)
                print(f"Stored {stop:,}/{len(frame):,} {split_name} examples", end="\r")
    print(f"Stored {len(frame):,} {split_name} examples in PostgreSQL.")

if DB_ENABLED:
    persist_dataset_split("train", train_raw)
    persist_dataset_split("test", test_raw)

RUN_ID = str(uuid.uuid4())
if DB_ENABLED:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """INSERT INTO experiment_runs (run_id, dataset_name, seed, device, status, config, notes)
                   VALUES (%s, %s, %s, %s, %s, %s, %s)""",
                (RUN_ID, DATASET_ID, SEED, str(DEVICE), "running", Json(CONFIG),
                 "Task 1: validation-selected traditional model, LSTM and BiLSTM finalist test evaluation."),
            )
print("Experiment run ID:", RUN_ID)
print("Raw data will be persisted online:", DB_ENABLED)

## 6. Minimal text normalization and dataset verification

Normalization is intentionally conservative: HTML entities/tags are normalized, URLs are replaced with a stable token, whitespace is collapsed, and text is lowercased. We do **not** remove all punctuation or stop words because those transformations can erase topic cues (for example, product names, abbreviations and technical terms). Vectorizers are fitted only on the training fold.

In [ ]:
def normalize_text(text: str) -> str:
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)                 # remove markup tags
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text, flags=re.IGNORECASE)
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

train_raw["text_clean"] = train_raw["text"].map(normalize_text)
test_raw["text_clean"] = test_raw["text"].map(normalize_text)


def describe_split(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = frame["label"].value_counts().reindex(LABEL_IDS, fill_value=0)
    result = pd.DataFrame({
        "label_id": LABEL_IDS,
        "category": LABEL_NAMES,
        "count": counts.values,
        "percentage": (counts.values / len(frame) * 100).round(2),
    })
    result.insert(0, "split", split_name)
    return result

class_distribution = pd.concat([
    describe_split(train_raw, "train"), describe_split(test_raw, "test")
], ignore_index=True)
display(class_distribution)
print("Empty raw train/test texts:", int(train_raw["text"].isna().sum()), int(test_raw["text"].isna().sum()))
print("Empty after normalization (train/test):",
      int((train_raw["text_clean"].str.len() == 0).sum()),
      int((test_raw["text_clean"].str.len() == 0).sum()))
print("Example source text:", train_raw.loc[0, "text"][:400])
print("Normalized model input:", train_raw.loc[0, "text_clean"][:400])

fig, ax = plt.subplots(figsize=(9, 4))
plot_dist = class_distribution.pivot(index="category", columns="split", values="count").reindex(LABEL_NAMES)
plot_dist.plot(kind="bar", ax=ax)
ax.set_title("AG News label distribution by official split")
ax.set_xlabel("")
ax.set_ylabel("Examples")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Split")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "plots" / "class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
class_distribution.to_csv(OUTPUT_DIR / "tables" / "class_distribution.csv", index=False)

## 7. Stratified train/validation split

The official training split stays separate from the official test split. A stratified 90/10 partition of the 120k training samples is used to compare models and select the recurrent epoch count. `source_index` continues to point to the original Hugging Face training row, allowing predictions/errors to be traced back to stored dataset examples.

In [ ]:
all_train_positions = np.arange(len(train_raw))
train_idx, val_idx = train_test_split(
    all_train_positions,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=train_raw["label"].to_numpy(),
)

X_train_texts = train_raw.loc[train_idx, "text_clean"].tolist()
y_train = train_raw.loc[train_idx, "label"].to_numpy(dtype=np.int64)
X_val_texts = train_raw.loc[val_idx, "text_clean"].tolist()
y_val = train_raw.loc[val_idx, "label"].to_numpy(dtype=np.int64)
X_full_texts = train_raw["text_clean"].tolist()
y_full = train_raw["label"].to_numpy(dtype=np.int64)
X_test_texts = test_raw["text_clean"].tolist()
y_test = test_raw["label"].to_numpy(dtype=np.int64)

print(f"Training fold: {len(train_idx):,} | Validation: {len(val_idx):,} | Official test: {len(test_raw):,}")
print("Training labels:", dict(zip(LABEL_NAMES, np.bincount(y_train, minlength=NUM_CLASSES))))
print("Validation labels:", dict(zip(LABEL_NAMES, np.bincount(y_val, minlength=NUM_CLASSES))))

# In-memory containers used by later comparison, reporting and persistence cells.
validation_results = []
validation_predictions = {}
training_histories = {}
test_results = []
test_predictions = {}
selected_model_configs = {}

## 8. Traditional representations and classifiers

We evaluate the full nine-combination grid:

- **BoW:** word unigram/bigram raw counts.
- **Word TF-IDF:** unigram/bigram TF-IDF with sublinear term frequency.
- **Character TF-IDF:** `char_wb` 3–5 character n-grams; useful for word fragments, names, punctuation and misspellings.
- **Algorithms:** Multinomial Naive Bayes, Logistic Regression, and Linear SVM.

Sparse matrices stay sparse. Each vectorizer is fitted on the training fold only and reused across the three classifiers for that representation. Training timings distinguish vectorizer fitting from classifier fitting.

In [ ]:
REPRESENTATIONS = {
    "bow": {
        "name": "Bag-of-Words",
        "factory": lambda: CountVectorizer(
            max_features=BOW_MAX_FEATURES, ngram_range=(1, 2), min_df=2,
            dtype=np.float32, lowercase=False, token_pattern=r"(?u)\b\w+\b"
        ),
    },
    "word_tfidf": {
        "name": "Word-level TF-IDF",
        "factory": lambda: TfidfVectorizer(
            max_features=WORD_TFIDF_MAX_FEATURES, ngram_range=(1, 2), min_df=2,
            sublinear_tf=True, strip_accents="unicode", dtype=np.float32,
            lowercase=False, token_pattern=r"(?u)\b\w+\b"
        ),
    },
    "char_tfidf": {
        "name": "Character-level TF-IDF",
        "factory": lambda: TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=3,
            max_features=CHAR_TFIDF_MAX_FEATURES, sublinear_tf=True,
            dtype=np.float32, lowercase=False
        ),
    },
}

CLASSIFIERS = {
    "naive_bayes": {
        "name": "Multinomial Naive Bayes",
        "factory": lambda: MultinomialNB(alpha=0.1),
        "config": {"alpha": 0.1},
    },
    "logistic_regression": {
        "name": "Logistic Regression",
        "factory": lambda: LogisticRegression(
            C=4.0, solver="saga", max_iter=100, tol=1e-3,
            n_jobs=-1, random_state=SEED
        ),
        "config": {"C": 4.0, "solver": "saga", "max_iter": 100, "tol": 1e-3, "n_jobs": -1},
    },
    "linear_svm": {
        "name": "Linear SVM (LinearSVC)",
        "factory": lambda: LinearSVC(C=1.0, max_iter=5000, dual="auto", random_state=SEED),
        "config": {"C": 1.0, "max_iter": 5000, "dual": "auto"},
    },
}


def calculate_metrics(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, labels=LABEL_IDS, target_names=LABEL_NAMES,
        output_dict=True, zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=LABEL_IDS, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=LABEL_IDS, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=LABEL_IDS, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=LABEL_IDS, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABEL_IDS).tolist(),
        "classification_report": report,
    }


def score_confidence(estimator, features):
    """Return max class score. SVM scores are softmax-normalized margins, not calibrated probabilities."""
    if hasattr(estimator, "predict_proba"):
        scores = estimator.predict_proba(features)
        kind = "predict_proba"
    else:
        raw = estimator.decision_function(features)
        if raw.ndim == 1:
            raw = np.column_stack([-raw, raw])
        scores = softmax(raw, axis=1)
        kind = "softmax_of_decision_scores_uncalibrated"
    return scores.max(axis=1).astype(np.float32), kind


traditional_validation_objects = {}
for representation_key, representation in REPRESENTATIONS.items():
    print(f"\n{'=' * 24} {representation['name']} {'=' * 24}")
    vectorizer = representation["factory"]()
    vectorizer_start = time.perf_counter()
    X_train_features = vectorizer.fit_transform(X_train_texts)
    X_val_features = vectorizer.transform(X_val_texts)
    vectorizer_seconds = time.perf_counter() - vectorizer_start
    print(f"Features: {X_train_features.shape[1]:,} | sparse nnz: {X_train_features.nnz:,} | vectorizer fit: {vectorizer_seconds:.2f}s")

    for classifier_key, classifier_spec in CLASSIFIERS.items():
        model_key = f"traditional__{representation_key}__{classifier_key}"
        estimator = classifier_spec["factory"]()
        fit_start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            estimator.fit(X_train_features, y_train)
        classifier_seconds = time.perf_counter() - fit_start
        y_pred = estimator.predict(X_val_features)
        confidence, confidence_kind = score_confidence(estimator, X_val_features)
        metrics = calculate_metrics(y_val, y_pred)
        result = {
            "model_key": model_key,
            "model_family": "traditional",
            "model_name": f"{representation['name']} + {classifier_spec['name']}",
            "representation": representation_key,
            "classifier": classifier_key,
            "training_seconds": float(vectorizer_seconds + classifier_seconds),
            "classifier_fit_seconds": float(classifier_seconds),
            "vectorizer_fit_seconds": float(vectorizer_seconds),
            "pipeline_fit_seconds": float(vectorizer_seconds + classifier_seconds),
            "confidence_kind": confidence_kind,
            "convergence_warning": any(issubclass(w.category, ConvergenceWarning) for w in caught_warnings),
            **metrics,
        }
        validation_results.append(result)
        validation_predictions[model_key] = {
            "source_indices": val_idx.copy(),
            "true_labels": y_val.copy(),
            "predicted_labels": np.asarray(y_pred, dtype=np.int64),
            "score_confidence": confidence,
            "confidence_kind": confidence_kind,
            "dataset_split": "train",
        }
        traditional_validation_objects[model_key] = {
            "representation": representation_key,
            "classifier": classifier_key,
        }
        print(f"{classifier_spec['name']:<26} Acc={result['accuracy']:.4f} Macro-F1={result['macro_f1']:.4f} fit={classifier_seconds:.1f}s")

    del X_train_features, X_val_features, vectorizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

traditional_validation_df = pd.DataFrame(validation_results).sort_values(
    ["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)
display(traditional_validation_df[[
    "model_name", "representation", "classifier", "accuracy", "macro_f1",
    "weighted_f1", "training_seconds", "vectorizer_fit_seconds", "pipeline_fit_seconds", "convergence_warning"
]])
traditional_validation_df.to_csv(OUTPUT_DIR / "tables" / "traditional_validation_results.csv", index=False)

## 9. LSTM and BiLSTM architecture

Both recurrent models share the same token vocabulary and training procedure during validation. Tokens are learned from the **training fold only**; unseen words map to `<UNK>` and padding maps to `<PAD>`. The architecture uses a trainable embedding layer, a 2-layer LSTM, dropout, and a linear four-class output. The BiLSTM concatenates the final forward and backward hidden states. Cross-entropy is optimized with AdamW, gradient clipping, early stopping on **validation Macro-F1**, and optional GPU mixed precision.

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)?|[^\w\s]", re.IGNORECASE)
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_ID = 0
UNK_ID = 1


def tokenize_text(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(str(text).lower())


def build_vocabulary(texts, max_size=MAX_VOCAB_SIZE, min_frequency=MIN_TOKEN_FREQUENCY):
    counts = Counter()
    for text in texts:
        counts.update(tokenize_text(text))
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
    for token, count in counts.most_common():
        if count < min_frequency:
            break
        if token in vocab:
            continue
        if len(vocab) >= max_size:
            break
        vocab[token] = len(vocab)
    return vocab


def encode_text(text: str, vocab: dict[str, int], max_length=MAX_SEQUENCE_LENGTH):
    token_ids = [vocab.get(tok, UNK_ID) for tok in tokenize_text(text)[:max_length]]
    if not token_ids:
        token_ids = [UNK_ID]
    return torch.tensor(token_ids, dtype=torch.long)


class NewsDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.sequences = [encode_text(text, vocab) for text in texts]
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        sequence = self.sequences[index]
        return sequence, int(self.labels[index]), len(sequence)


def collate_news_batch(batch):
    sequences, labels, lengths = zip(*batch)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    return padded, torch.tensor(labels, dtype=torch.long), torch.tensor(lengths, dtype=torch.long)


class RecurrentTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=LSTM_LAYERS, dropout=DROPOUT, bidirectional=False, num_classes=NUM_CLASSES):
        super().__init__()
        self.bidirectional = bool(bidirectional)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.embedding_dropout = nn.Dropout(dropout * 0.5)
        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=self.bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        representation_dim = hidden_dim * (2 if self.bidirectional else 1)
        self.classifier_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(representation_dim, num_classes)

    def forward(self, token_ids, lengths):
        embedded = self.embedding_dropout(self.embedding(token_ids))
        packed = pack_padded_sequence(
            embedded, lengths.detach().cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.encoder(packed)
        if self.bidirectional:
            final_representation = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            final_representation = hidden[-1]
        return self.classifier(self.classifier_dropout(final_representation))


vocab_fold = build_vocabulary(X_train_texts)
print(f"Vocabulary size (training fold only): {len(vocab_fold):,}")

train_nn_dataset = NewsDataset(X_train_texts, y_train, vocab_fold)
val_nn_dataset = NewsDataset(X_val_texts, y_val, vocab_fold)
train_loader = DataLoader(
    train_nn_dataset, batch_size=NN_BATCH_SIZE, shuffle=True, collate_fn=collate_news_batch,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_nn_dataset, batch_size=NN_BATCH_SIZE, shuffle=False, collate_fn=collate_news_batch,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)

print("Architecture configuration:", {
    "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM, "layers": LSTM_LAYERS,
    "dropout": DROPOUT, "classes": NUM_CLASSES, "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "device": str(DEVICE), "mixed_precision": USE_AMP,
})

In [ ]:
def evaluate_recurrent_model(model, loader):
    model.eval()
    all_true, all_pred, all_confidence, losses = [], [], [], []
    criterion = nn.CrossEntropyLoss()
    with torch.inference_mode():
        for token_ids, labels, lengths in loader:
            token_ids = token_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(token_ids, lengths)
                loss = criterion(logits, labels)
            probabilities = torch.softmax(logits.float(), dim=1)
            predictions = probabilities.argmax(dim=1)
            losses.append(float(loss.item()) * len(labels))
            all_true.extend(labels.cpu().numpy().tolist())
            all_pred.extend(predictions.cpu().numpy().tolist())
            all_confidence.extend(probabilities.max(dim=1).values.cpu().numpy().tolist())
    metrics = calculate_metrics(np.asarray(all_true), np.asarray(all_pred))
    metrics["loss"] = float(sum(losses) / max(1, len(all_true)))
    return metrics, np.asarray(all_pred, dtype=np.int64), np.asarray(all_confidence, dtype=np.float32)


def fit_recurrent_with_early_stopping(model_key: str, bidirectional: bool):
    model = RecurrentTextClassifier(vocab_size=len(vocab_fold), bidirectional=bidirectional).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NN_LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_macro_f1 = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    history = []
    total_start = time.perf_counter()

    for epoch in range(1, NN_EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for token_ids, labels, lengths in train_loader:
            token_ids = token_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(token_ids, lengths)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item()) * len(labels)
            seen += len(labels)

        train_loss = running_loss / max(1, seen)
        val_metrics, _, _ = evaluate_recurrent_model(model, val_loader)
        epoch_seconds = time.perf_counter() - epoch_start
        learning_rate = float(optimizer.param_groups[0]["lr"])
        history.append({
            "model_key": model_key, "training_stage": "selection_training", "epoch": epoch,
            "train_loss": train_loss, "validation_loss": val_metrics["loss"],
            "validation_accuracy": val_metrics["accuracy"], "validation_macro_f1": val_metrics["macro_f1"],
            "learning_rate": learning_rate, "epoch_seconds": epoch_seconds,
        })
        print(f"{model_key} | epoch {epoch:02d}/{NN_EPOCHS} | train_loss={train_loss:.4f} "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
              f"val_macro_f1={val_metrics['macro_f1']:.4f} | {epoch_seconds:.1f}s")

        if val_metrics["macro_f1"] > best_macro_f1 + 1e-5:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= NN_PATIENCE:
                print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
                break

    total_seconds = time.perf_counter() - total_start
    if best_state is None:
        raise RuntimeError(f"Training produced no checkpoint for {model_key}")
    model.load_state_dict(best_state)
    model.to(DEVICE)
    best_val_metrics, val_pred, val_conf = evaluate_recurrent_model(model, val_loader)
    result = {
        "model_key": model_key,
        "model_family": "recurrent_neural_network",
        "model_name": "BiLSTM" if bidirectional else "LSTM",
        "representation": "learned_word_embeddings",
        "classifier": "BiLSTM" if bidirectional else "LSTM",
        "training_seconds": float(total_seconds),
        "vectorizer_fit_seconds": 0.0,
        "pipeline_fit_seconds": float(total_seconds),
        "confidence_kind": "softmax_probability",
        "best_epoch": int(best_epoch),
        "vocab_size": len(vocab_fold),
        **{k: v for k, v in best_val_metrics.items() if k != "loss"},
        "validation_loss": best_val_metrics["loss"],
    }
    validation_predictions[model_key] = {
        "source_indices": val_idx.copy(),
        "true_labels": y_val.copy(),
        "predicted_labels": val_pred,
        "score_confidence": val_conf,
        "confidence_kind": "softmax_probability",
        "dataset_split": "train",
    }
    training_histories[model_key] = pd.DataFrame(history)
    selected_model_configs[model_key] = {
        "model_family": "recurrent_neural_network", "bidirectional": bidirectional,
        "best_epoch": best_epoch, "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM,
        "layers": LSTM_LAYERS, "dropout": DROPOUT, "learning_rate": NN_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "batch_size": NN_BATCH_SIZE,
        "max_sequence_length": MAX_SEQUENCE_LENGTH, "vocab_size": len(vocab_fold),
    }
    torch.save(best_state, OUTPUT_DIR / "models" / f"{model_key}_validation_best.pt")
    return result


for model_key, bidirectional in [("lstm", False), ("bilstm", True)]:
    result = fit_recurrent_with_early_stopping(model_key, bidirectional)
    validation_results.append(result)
    print(f"Completed {result['model_name']}: validation Macro-F1={result['macro_f1']:.4f}; best epoch={result['best_epoch']}")

nn_validation_df = pd.DataFrame([r for r in validation_results if r["model_family"] == "recurrent_neural_network"])
display(nn_validation_df[["model_name", "accuracy", "macro_f1", "weighted_f1", "training_seconds", "best_epoch", "vocab_size"]])

## 10. Validation comparison and model selection

**Primary selection metric: Macro-F1.** Accuracy is also reported, but Macro-F1 weights all four categories equally and is a more informative comparison if class balance changes. Training times are measured in the active Kaggle session and will depend on CPU/GPU type and software versions. The traditional end-to-end fitting time includes feature-vectorizer fit plus classifier fit; classifier-only time is also shown for transparency.

In [ ]:
validation_df = pd.DataFrame(validation_results).sort_values(
    ["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)
validation_df.insert(0, "validation_rank", np.arange(1, len(validation_df) + 1))

# Select only from validation scores. The test split is not read in this selection step.
traditional_candidates = validation_df[validation_df["model_family"] == "traditional"]
best_traditional_row = traditional_candidates.sort_values(
    ["macro_f1", "accuracy"], ascending=False
).iloc[0]
selected_traditional_key = str(best_traditional_row["model_key"])

print("Validation ranking (all 11 configurations):")
display(validation_df[[
    "validation_rank", "model_name", "model_family", "representation", "accuracy", "macro_f1",
    "weighted_f1", "training_seconds", "vectorizer_fit_seconds", "pipeline_fit_seconds"
]])
print("\nTraditional model selected on validation:", best_traditional_row["model_name"])
print(f"Validation Accuracy={best_traditional_row['accuracy']:.4f} | Macro-F1={best_traditional_row['macro_f1']:.4f}")

validation_df.to_csv(OUTPUT_DIR / "tables" / "validation_model_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
ordered = validation_df.sort_values("macro_f1", ascending=True)
ax.barh(ordered["model_name"], ordered["macro_f1"])
ax.set_xlabel("Validation Macro-F1")
ax.set_title("Validation Macro-F1 — all traditional and recurrent models")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "plots" / "validation_macro_f1_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
time_ordered = validation_df.sort_values("pipeline_fit_seconds" if "pipeline_fit_seconds" in validation_df else "training_seconds", ascending=True)
ax.barh(time_ordered["model_name"], time_ordered["training_seconds"])
ax.set_xlabel("Fit time (seconds; recurrent time includes validation passes)")
ax.set_title("Training-time comparison (measured in this runtime)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "plots" / "validation_training_time_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

for key, history in training_histories.items():
    history.to_csv(OUTPUT_DIR / "tables" / f"{key}_selection_training_history.csv", index=False)

## 11. Refit the selected traditional pipeline and evaluate on the official test set

The vectorizer and classifier are refitted on **all official training examples**. Only the validation-selected traditional combination is run on the official test set. This gives a final evaluation on the maximum available labeled training data without choosing a winner by test scores.

In [ ]:
selected_info = traditional_validation_objects[selected_traditional_key]
selected_representation = selected_info["representation"]
selected_classifier = selected_info["classifier"]
selected_vectorizer = REPRESENTATIONS[selected_representation]["factory"]()

start = time.perf_counter()
X_full_features = selected_vectorizer.fit_transform(X_full_texts)
vectorizer_final_seconds = time.perf_counter() - start
selected_estimator = CLASSIFIERS[selected_classifier]["factory"]()
start = time.perf_counter()
with warnings.catch_warnings(record=True) as final_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    selected_estimator.fit(X_full_features, y_full)
classifier_final_seconds = time.perf_counter() - start
X_test_features = selected_vectorizer.transform(X_test_texts)
y_test_pred = selected_estimator.predict(X_test_features)
y_test_conf, test_confidence_kind = score_confidence(selected_estimator, X_test_features)
traditional_test_metrics = calculate_metrics(y_test, y_test_pred)
traditional_test_result = {
    "model_key": selected_traditional_key,
    "model_family": "traditional",
    "model_name": str(best_traditional_row["model_name"]),
    "representation": selected_representation,
    "classifier": selected_classifier,
    "training_seconds": float(classifier_final_seconds + vectorizer_final_seconds),
    "classifier_fit_seconds": float(classifier_final_seconds),
    "vectorizer_fit_seconds": float(vectorizer_final_seconds),
    "pipeline_fit_seconds": float(classifier_final_seconds + vectorizer_final_seconds),
    "confidence_kind": test_confidence_kind,
    "convergence_warning": any(issubclass(w.category, ConvergenceWarning) for w in final_warnings),
    **traditional_test_metrics,
}
test_results.append(traditional_test_result)
test_predictions[selected_traditional_key] = {
    "source_indices": test_raw["source_index"].to_numpy(dtype=np.int64),
    "true_labels": y_test.copy(),
    "predicted_labels": np.asarray(y_test_pred, dtype=np.int64),
    "score_confidence": y_test_conf,
    "confidence_kind": test_confidence_kind,
    "dataset_split": "test",
}
selected_model_configs[selected_traditional_key] = {
    "model_family": "traditional", "representation": selected_representation,
    "classifier": selected_classifier,
    "vectorizer_config": {k: v for k, v in REPRESENTATIONS[selected_representation].items() if k != "factory"},
    "classifier_config": CLASSIFIERS[selected_classifier]["config"],
}
joblib.dump({"vectorizer": selected_vectorizer, "estimator": selected_estimator,
             "label_names": LABEL_NAMES, "model_key": selected_traditional_key},
            OUTPUT_DIR / "models" / "best_traditional_pipeline.joblib", compress=3)

print("Selected traditional final test result:")
print(f"Accuracy={traditional_test_result['accuracy']:.4f} | Macro-F1={traditional_test_result['macro_f1']:.4f} | "
      f"vectorizer={vectorizer_final_seconds:.1f}s | classifier={classifier_final_seconds:.1f}s")
print("Convergence warning:", traditional_test_result["convergence_warning"])
print(classification_report(y_test, y_test_pred, labels=LABEL_IDS, target_names=LABEL_NAMES, digits=4, zero_division=0))

# Release sparse feature matrices after the final traditional evaluation.
del X_full_features, X_test_features

## 12. Refit LSTM and BiLSTM on all official training examples, then test

For each recurrent architecture, use the best epoch count from validation early stopping, rebuild its vocabulary using all official training text, train a fresh model on the full official training split for exactly that many epochs, and evaluate once on the test split. The final refit does not monitor or adapt to test performance.

In [ ]:
def fit_recurrent_fixed_epochs(model_key: str, bidirectional: bool, epochs: int):
    vocab_full = build_vocabulary(X_full_texts)
    full_dataset = NewsDataset(X_full_texts, y_full, vocab_full)
    full_loader = DataLoader(
        full_dataset, batch_size=NN_BATCH_SIZE, shuffle=True, collate_fn=collate_news_batch,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
    )
    test_dataset = NewsDataset(X_test_texts, y_test, vocab_full)
    final_test_loader = DataLoader(
        test_dataset, batch_size=NN_BATCH_SIZE, shuffle=False, collate_fn=collate_news_batch,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
    )

    seed_everything(SEED)
    model = RecurrentTextClassifier(vocab_size=len(vocab_full), bidirectional=bidirectional).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NN_LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = []
    total_start = time.perf_counter()

    for epoch in range(1, int(epochs) + 1):
        model.train()
        total_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for token_ids, labels, lengths in full_loader:
            token_ids = token_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(token_ids, lengths)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item()) * len(labels)
            seen += len(labels)
        epoch_seconds = time.perf_counter() - epoch_start
        history.append({
            "model_key": model_key, "training_stage": "final_refit", "epoch": epoch,
            "train_loss": total_loss / max(1, seen), "validation_loss": None,
            "validation_accuracy": None, "validation_macro_f1": None,
            "learning_rate": float(optimizer.param_groups[0]["lr"]), "epoch_seconds": epoch_seconds,
        })
        print(f"{model_key} final-refit epoch {epoch:02d}/{epochs} | "
              f"train_loss={history[-1]['train_loss']:.4f} | {epoch_seconds:.1f}s")

    total_seconds = time.perf_counter() - total_start
    test_metrics, pred, confidence = evaluate_recurrent_model(model, final_test_loader)
    result = {
        "model_key": model_key,
        "model_family": "recurrent_neural_network",
        "model_name": "BiLSTM" if bidirectional else "LSTM",
        "representation": "learned_word_embeddings",
        "classifier": "BiLSTM" if bidirectional else "LSTM",
        "training_seconds": float(total_seconds),
        "vectorizer_fit_seconds": 0.0,
        "pipeline_fit_seconds": float(total_seconds),
        "confidence_kind": "softmax_probability",
        "best_epoch": int(epochs),
        "vocab_size": len(vocab_full),
        **{k: v for k, v in test_metrics.items() if k != "loss"},
        "test_loss": test_metrics["loss"],
    }
    test_predictions[model_key] = {
        "source_indices": test_raw["source_index"].to_numpy(dtype=np.int64),
        "true_labels": y_test.copy(),
        "predicted_labels": pred,
        "score_confidence": confidence,
        "confidence_kind": "softmax_probability",
        "dataset_split": "test",
    }
    training_histories[f"{model_key}_final_refit"] = pd.DataFrame(history)
    model_path = OUTPUT_DIR / "models" / f"{model_key}_final.pt"
    torch.save({
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "vocab": vocab_full,
        "label_names": LABEL_NAMES,
        "config": {
            "embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM, "num_layers": LSTM_LAYERS,
            "dropout": DROPOUT, "bidirectional": bidirectional,
            "max_sequence_length": MAX_SEQUENCE_LENGTH, "vocab_size": len(vocab_full),
        },
    }, model_path)
    with open(OUTPUT_DIR / "models" / f"{model_key}_vocab.json", "w", encoding="utf-8") as f:
        json.dump(vocab_full, f, ensure_ascii=False)
    selected_model_configs[model_key] = {
        "model_family": "recurrent_neural_network", "bidirectional": bidirectional,
        "epochs_final_refit": int(epochs), "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM, "layers": LSTM_LAYERS, "dropout": DROPOUT,
        "learning_rate": NN_LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "batch_size": NN_BATCH_SIZE, "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "vocab_size": len(vocab_full),
    }
    del full_dataset, full_loader, test_dataset, final_test_loader
    return result


for model_key, bidirectional in [("lstm", False), ("bilstm", True)]:
    selection_row = next(r for r in validation_results if r["model_key"] == model_key)
    final_result = fit_recurrent_fixed_epochs(model_key, bidirectional, int(selection_row["best_epoch"]))
    test_results.append(final_result)
    print(f"{final_result['model_name']} test: Accuracy={final_result['accuracy']:.4f}, Macro-F1={final_result['macro_f1']:.4f}")

# Save selection and final-refit histories for later plots/analysis.
for history_key, history_df in training_histories.items():
    history_df.to_csv(OUTPUT_DIR / "tables" / f"{history_key}_history.csv", index=False)

## 13. Final holdout test results and per-class evaluation

This table has final **official test** results for the validation-selected traditional model, LSTM and BiLSTM. The broader 11-model comparison is the validation table above, which was used for selection. Keep those roles distinct in the report.

In [ ]:
test_results_df = pd.DataFrame(test_results).sort_values(["macro_f1", "accuracy"], ascending=False).reset_index(drop=True)
test_results_df.insert(0, "test_rank_for_reporting_only", np.arange(1, len(test_results_df) + 1))
display(test_results_df[[
    "test_rank_for_reporting_only", "model_name", "model_family", "representation",
    "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall",
    "training_seconds", "vectorizer_fit_seconds", "best_epoch"
]])

test_results_df.to_csv(OUTPUT_DIR / "tables" / "final_test_results.csv", index=False)
with open(OUTPUT_DIR / "tables" / "final_test_classification_reports.json", "w", encoding="utf-8") as f:
    json.dump({r["model_key"]: r["classification_report"] for r in test_results}, f, indent=2, ensure_ascii=False)

per_class_rows = []
for result in test_results:
    report = result["classification_report"]
    for category in LABEL_NAMES:
        row = report[category]
        per_class_rows.append({
            "model_key": result["model_key"], "model_name": result["model_name"],
            "category": category, "precision": row["precision"], "recall": row["recall"],
            "f1_score": row["f1-score"], "support": int(row["support"]),
        })
per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)
per_class_df.to_csv(OUTPUT_DIR / "tables" / "per_class_test_metrics.csv", index=False)

# Include the validation-based selected traditional model and both recurrent models.
for model_key, result in [(r["model_key"], r) for r in test_results]:
    cm = np.asarray(result["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABEL_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"Official test confusion matrix — {result['model_name']}")
    fig.tight_layout()
    safe_key = re.sub(r"[^a-zA-Z0-9_-]+", "_", model_key)
    fig.savefig(OUTPUT_DIR / "plots" / f"confusion_matrix_{safe_key}.png", dpi=180, bbox_inches="tight")
    plt.show()

# Merge validation and final test numbers without implying that all models were selected on test.
val_for_merge = validation_df[["model_key", "model_name", "accuracy", "macro_f1", "training_seconds"]].rename(columns={
    "accuracy": "validation_accuracy", "macro_f1": "validation_macro_f1", "training_seconds": "validation_fit_seconds"
})
test_for_merge = test_results_df[["model_key", "accuracy", "macro_f1", "training_seconds"]].rename(columns={
    "accuracy": "official_test_accuracy", "macro_f1": "official_test_macro_f1", "training_seconds": "final_refit_seconds"
})
comparison_df = val_for_merge.merge(test_for_merge, on="model_key", how="left")
display(comparison_df.sort_values("validation_macro_f1", ascending=False))
comparison_df.to_csv(OUTPUT_DIR / "tables" / "validation_and_test_comparison.csv", index=False)

## 14. Error analysis: where does each finalist fail?

The cell below creates:

1. **Confusion-pair counts** (for example, Business → World),
2. per-model mistake tables with the source index, true/predicted category, model score and original text,
3. a compact high-score error sample to inspect likely overconfident mistakes.

For the SVM, the displayed score is the maximum softmax-normalized decision margin and is **not calibrated probability**. LSTM/BiLSTM and models exposing `predict_proba` report their model probabilities; those may still be miscalibrated.

In [ ]:
error_frames = []
confusion_pair_frames = []

for result in test_results:
    model_key = result["model_key"]
    pred_bundle = test_predictions[model_key]
    frame = pd.DataFrame({
        "model_key": model_key,
        "model_name": result["model_name"],
        "source_index": pred_bundle["source_indices"],
        "true_label": pred_bundle["true_labels"],
        "predicted_label": pred_bundle["predicted_labels"],
        "score_confidence": pred_bundle["score_confidence"],
        "confidence_kind": pred_bundle["confidence_kind"],
    })
    frame["true_category"] = frame["true_label"].map(dict(enumerate(LABEL_NAMES)))
    frame["predicted_category"] = frame["predicted_label"].map(dict(enumerate(LABEL_NAMES)))
    frame["is_correct"] = frame["true_label"] == frame["predicted_label"]
    frame["text"] = frame["source_index"].map(test_raw.set_index("source_index")["text"].to_dict())
    frame["text_clean"] = frame["source_index"].map(test_raw.set_index("source_index")["text_clean"].to_dict())
    errors = frame.loc[~frame["is_correct"]].copy()
    error_frames.append(errors)
    pairs = errors.groupby(["true_category", "predicted_category"]).size().reset_index(name="count")
    pairs["model_key"] = model_key
    pairs["model_name"] = result["model_name"]
    confusion_pair_frames.append(pairs)

all_errors_df = pd.concat(error_frames, ignore_index=True) if error_frames else pd.DataFrame()
confusion_pairs_df = pd.concat(confusion_pair_frames, ignore_index=True) if confusion_pair_frames else pd.DataFrame()

print("Most common cross-category confusions per finalist:")
for model_key, group in confusion_pairs_df.groupby("model_key"):
    print("\n", group["model_name"].iloc[0])
    display(group.sort_values("count", ascending=False).head(8))

# Export all mistakes; useful for manual annotation/review with your teammate.
all_errors_df.to_csv(OUTPUT_DIR / "tables" / "all_test_errors.csv", index=False)
confusion_pairs_df.to_csv(OUTPUT_DIR / "tables" / "test_confusion_pairs.csv", index=False)

# A useful inspection sample: highest score among mistakes, with confidence semantics shown.
high_score_errors = (
    all_errors_df.sort_values(["model_key", "score_confidence"], ascending=[True, False])
    .groupby("model_key", group_keys=False).head(12)
    .sort_values(["model_name", "score_confidence"], ascending=[True, False])
)
error_display_cols = ["model_name", "source_index", "true_category", "predicted_category", "score_confidence", "confidence_kind", "text"]
display(high_score_errors[error_display_cols].head(36))
high_score_errors[error_display_cols].to_csv(OUTPUT_DIR / "tables" / "high_score_test_errors.csv", index=False)

# Qualitative check: a few errors per actual class for each finalist.
representative = (
    all_errors_df.groupby(["model_key", "true_category"], group_keys=False)
    .head(3)[error_display_cols]
)
representative.to_csv(OUTPUT_DIR / "tables" / "representative_test_errors.csv", index=False)
display(representative.head(30))

## 15. Learning curves and training diagnostics

Selection-stage curves show whether each recurrent model was still improving, plateauing, or overfitting when early stopping selected its checkpoint. Final-refit curves contain training loss only because the final refit is deliberately not monitored against validation/test data.

In [ ]:
for model_key in ["lstm", "bilstm"]:
    history = training_histories.get(model_key)
    if history is None or history.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train loss")
    ax.plot(history["epoch"], history["validation_loss"], marker="o", label="Validation loss")
    ax.set_title(f"Loss curves — {model_key.upper()} validation/selection training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "plots" / f"{model_key}_learning_curve.png", dpi=180, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["epoch"], history["validation_macro_f1"], marker="o")
    ax.set_ylim(0, 1)
    ax.set_title(f"Validation Macro-F1 — {model_key.upper()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Macro-F1")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "plots" / f"{model_key}_validation_macro_f1.png", dpi=180, bbox_inches="tight")
    plt.show()

## 16. Persist experiment metrics, histories and predictions to PostgreSQL

If PostgreSQL is configured, validation records for all 11 models are saved, along with predictions for the validation examples. The final test metrics/predictions are saved only for the selected traditional pipeline, LSTM and BiLSTM. Predictions reference stored examples through their original Hugging Face source index.

In [ ]:
def persist_metrics_and_predictions(stage: str, model_key: str, model_family: str, representation,
                                    metrics: dict, training_seconds, vectorizer_seconds, model_config: dict,
                                    prediction_bundle: dict | None = None, best_epoch: int | None = None):
    if not DB_ENABLED:
        return
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """INSERT INTO model_metrics (
                    run_id, evaluation_stage, model_key, model_family, representation,
                    accuracy, macro_f1, weighted_f1, macro_precision, macro_recall,
                    training_seconds, vectorizer_fit_seconds, best_epoch,
                    confusion_matrix, classification_report, model_config
                ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (run_id, evaluation_stage, model_key) DO UPDATE SET
                    accuracy=EXCLUDED.accuracy, macro_f1=EXCLUDED.macro_f1,
                    weighted_f1=EXCLUDED.weighted_f1, macro_precision=EXCLUDED.macro_precision,
                    macro_recall=EXCLUDED.macro_recall, training_seconds=EXCLUDED.training_seconds,
                    vectorizer_fit_seconds=EXCLUDED.vectorizer_fit_seconds, best_epoch=EXCLUDED.best_epoch,
                    confusion_matrix=EXCLUDED.confusion_matrix,
                    classification_report=EXCLUDED.classification_report, model_config=EXCLUDED.model_config""",
                (RUN_ID, stage, model_key, model_family, representation,
                 float(metrics["accuracy"]), float(metrics["macro_f1"]), float(metrics["weighted_f1"]),
                 float(metrics["macro_precision"]), float(metrics["macro_recall"]),
                 None if training_seconds is None else float(training_seconds),
                 None if vectorizer_seconds is None else float(vectorizer_seconds),
                 best_epoch, Json(metrics["confusion_matrix"]), Json(metrics["classification_report"]),
                 Json(model_config)),
            )
            if prediction_bundle is not None:
                ds_split = "train" if stage == "validation" else "test"
                rows = []
                for source_index, true_label, predicted_label, confidence in zip(
                    prediction_bundle["source_indices"], prediction_bundle["true_labels"],
                    prediction_bundle["predicted_labels"], prediction_bundle["score_confidence"]
                ):
                    rows.append((
                        RUN_ID, stage, model_key, DATASET_ID, ds_split, int(source_index),
                        int(true_label), int(predicted_label), float(confidence),
                        bool(int(true_label) == int(predicted_label)),
                    ))
                sql = """INSERT INTO model_predictions (
                    run_id, evaluation_stage, model_key, dataset_name, dataset_split, source_index,
                    true_label, predicted_label, score_confidence, is_correct
                ) VALUES %s ON CONFLICT DO NOTHING"""
                for start in range(0, len(rows), 2000):
                    execute_values(cur, sql, rows[start:start + 2000], page_size=500)


def persist_history(history_frame: pd.DataFrame, model_key: str, training_stage: str) -> None:
    if not DB_ENABLED or history_frame is None or history_frame.empty:
        return
    sql = """INSERT INTO training_history (
        run_id, model_key, training_stage, epoch, train_loss, validation_loss,
        validation_accuracy, validation_macro_f1, learning_rate
    ) VALUES %s ON CONFLICT (run_id, model_key, training_stage, epoch) DO UPDATE SET
        train_loss=EXCLUDED.train_loss, validation_loss=EXCLUDED.validation_loss,
        validation_accuracy=EXCLUDED.validation_accuracy,
        validation_macro_f1=EXCLUDED.validation_macro_f1, learning_rate=EXCLUDED.learning_rate"""
    rows = []
    for _, row in history_frame.iterrows():
        def optional_float(value):
            return None if pd.isna(value) else float(value)
        rows.append((RUN_ID, model_key, training_stage, int(row["epoch"]),
                     optional_float(row.get("train_loss")), optional_float(row.get("validation_loss")),
                     optional_float(row.get("validation_accuracy")), optional_float(row.get("validation_macro_f1")),
                     optional_float(row.get("learning_rate"))))
    with db_connect() as conn:
        with conn.cursor() as cur:
            for start in range(0, len(rows), 1000):
                execute_values(cur, sql, rows[start:start + 1000], page_size=500)

if DB_ENABLED:
    # Validation results and validation predictions for the complete 11-model comparison.
    for result in validation_results:
        model_key = result["model_key"]
        representation = result.get("representation")
        model_config = selected_model_configs.get(model_key, {})
        if result["model_family"] == "traditional":
            model_config = {
                "representation": result["representation"],
                "classifier": result["classifier"],
                "classifier_config": CLASSIFIERS[result["classifier"]]["config"],
                "confidence_kind": result.get("confidence_kind"),
            }
        metrics_payload = {k: result[k] for k in [
            "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall",
            "confusion_matrix", "classification_report"
        ]}
        persist_metrics_and_predictions(
            "validation", model_key, result["model_family"], representation,
            metrics_payload, result.get("training_seconds"), result.get("vectorizer_fit_seconds"),
            model_config, validation_predictions[model_key], result.get("best_epoch")
        )
    for result in test_results:
        model_key = result["model_key"]
        metrics_payload = {k: result[k] for k in [
            "accuracy", "macro_f1", "weighted_f1", "macro_precision", "macro_recall",
            "confusion_matrix", "classification_report"
        ]}
        persist_metrics_and_predictions(
            "test", model_key, result["model_family"], result.get("representation"),
            metrics_payload, result.get("training_seconds"), result.get("vectorizer_fit_seconds"),
            selected_model_configs.get(model_key, {}), test_predictions[model_key], result.get("best_epoch")
        )
    for history_key, history_df in training_histories.items():
        if history_key.endswith("_final_refit"):
            base_model_key = history_key.removesuffix("_final_refit")
            persist_history(history_df, base_model_key, "final_refit")
        else:
            persist_history(history_df, history_key, "selection_training")
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute("UPDATE experiment_runs SET status=%s, completed_at=NOW() WHERE run_id=%s", ("completed", RUN_ID))
    print("Persisted dataset examples, run metadata, metrics, predictions, and histories to PostgreSQL.")
else:
    print("Skipped PostgreSQL persistence; local CSV/JSON/model artifacts were still saved.")

## 17. Export a compact run summary and all artifacts

The output directory is `/kaggle/working/ag_news_task1_outputs/` on Kaggle. Download the directory from the notebook's Output panel or zip it using the final cell. The saved files include metrics/tables, figures, the best traditional pipeline, final LSTM/BiLSTM weights, vocabularies and JSON metadata.

In [ ]:
run_summary = {
    "run_id": RUN_ID,
    "dataset_id": DATASET_ID,
    "source_url": "https://huggingface.co/datasets/fancyzhx/ag_news",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_examples": int(len(train_raw)),
    "test_examples": int(len(test_raw)),
    "label_names": LABEL_NAMES,
    "seed": SEED,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "postgres_persistence_enabled": DB_ENABLED,
    "selected_traditional_model": selected_traditional_key,
    "validation_best_traditional": {
        "accuracy": float(best_traditional_row["accuracy"]),
        "macro_f1": float(best_traditional_row["macro_f1"]),
    },
    "validation_comparison": validation_df[["model_key", "model_name", "accuracy", "macro_f1", "training_seconds"]].to_dict(orient="records"),
    "official_test_results": test_results_df[["model_key", "model_name", "accuracy", "macro_f1", "weighted_f1", "training_seconds"]].to_dict(orient="records"),
    "config": CONFIG,
}
with open(OUTPUT_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False, default=lambda o: float(o) if isinstance(o, np.floating) else int(o) if isinstance(o, np.integer) else str(o))

# Machine-readable predictions, not only rendered notebook output.
for model_key, bundle in validation_predictions.items():
    if model_key.startswith("traditional__") or model_key in ("lstm", "bilstm"):
        frame = pd.DataFrame({
            "source_index": bundle["source_indices"],
            "true_label": bundle["true_labels"],
            "predicted_label": bundle["predicted_labels"],
            "score_confidence": bundle["score_confidence"],
            "is_correct": np.asarray(bundle["true_labels"]) == np.asarray(bundle["predicted_labels"]),
        })
        frame["true_category"] = frame["true_label"].map(dict(enumerate(LABEL_NAMES)))
        frame["predicted_category"] = frame["predicted_label"].map(dict(enumerate(LABEL_NAMES)))
        frame.to_csv(OUTPUT_DIR / "tables" / f"validation_predictions_{model_key}.csv", index=False)

for model_key, bundle in test_predictions.items():
    frame = pd.DataFrame({
        "source_index": bundle["source_indices"],
        "true_label": bundle["true_labels"],
        "predicted_label": bundle["predicted_labels"],
        "score_confidence": bundle["score_confidence"],
        "confidence_kind": bundle["confidence_kind"],
        "is_correct": np.asarray(bundle["true_labels"]) == np.asarray(bundle["predicted_labels"]),
    })
    frame["true_category"] = frame["true_label"].map(dict(enumerate(LABEL_NAMES)))
    frame["predicted_category"] = frame["predicted_label"].map(dict(enumerate(LABEL_NAMES)))
    frame.to_csv(OUTPUT_DIR / "tables" / f"test_predictions_{model_key}.csv", index=False)

print("Output directory:", OUTPUT_DIR.resolve())
print("Top-level artifacts:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")

## 18. Inference helpers for the saved finalist models

These helpers let you try new text after training. The classical model is packaged as a scikit-learn pipeline dictionary; the neural checkpoint includes the vocabulary and architecture configuration. The functions below expect the final-refit checkpoints saved by this notebook.

In [ ]:
def predict_with_traditional_pipeline(text: str):
    artifact = joblib.load(OUTPUT_DIR / "models" / "best_traditional_pipeline.joblib")
    cleaned = normalize_text(text)
    vector = artifact["vectorizer"].transform([cleaned])
    estimator = artifact["estimator"]
    prediction = int(estimator.predict(vector)[0])
    confidence, confidence_kind = score_confidence(estimator, vector)
    return {
        "category": artifact["label_names"][prediction],
        "label_id": prediction,
        "score_confidence": float(confidence[0]),
        "confidence_kind": confidence_kind,
    }


def load_recurrent_artifact(model_key: str):
    checkpoint_path = OUTPUT_DIR / "models" / f"{model_key}_final.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg = checkpoint["config"]
    model = RecurrentTextClassifier(
        vocab_size=len(checkpoint["vocab"]), embedding_dim=cfg["embedding_dim"],
        hidden_dim=cfg["hidden_dim"], num_layers=cfg["num_layers"], dropout=cfg["dropout"],
        bidirectional=cfg["bidirectional"], num_classes=len(checkpoint["label_names"])
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.to(DEVICE).eval()
    return model, checkpoint["vocab"], checkpoint["label_names"], cfg


def predict_with_recurrent_model(text: str, model_key: str = "bilstm"):
    model, vocab, label_names, cfg = load_recurrent_artifact(model_key)
    sequence = encode_text(normalize_text(text), vocab, cfg["max_sequence_length"]).unsqueeze(0).to(DEVICE)
    lengths = torch.tensor([sequence.shape[1]], dtype=torch.long)
    with torch.inference_mode():
        logits = model(sequence, lengths)
        probabilities = torch.softmax(logits.float(), dim=1)[0]
    label_id = int(probabilities.argmax().item())
    return {
        "category": label_names[label_id],
        "label_id": label_id,
        "probabilities": {name: float(probabilities[i].item()) for i, name in enumerate(label_names)},
    }

# Example calls (replace with any article headline/description after you execute all preceding cells):
# predict_with_traditional_pipeline("Central bank holds interest rates as inflation outlook improves")
# predict_with_recurrent_model("Scientists report a new observation from a space telescope", model_key="bilstm")

## 19. Report-writing checklist and interpretation guidance

When writing your results section, distinguish **validation model selection** from **official test evaluation**. Do not invent results: copy figures/tables from the executed run.

Suggested discussion points:

- Which feature representation/algorithm combination achieved the highest validation Macro-F1? How much slower was its full pipeline compared with the alternatives?
- Did word bigrams help distinguish overlapping topics? Did character n-grams help with names, abbreviations, and technical vocabulary?
- Did the recurrent models improve on sparse linear baselines? Compare test Accuracy and Macro-F1 together with wall-clock training time, not only a single score.
- Which class pairs have the highest off-diagonal confusion counts? Inspect the associated examples. Are they genuinely ambiguous, contain multiple topics, or seem mislabeled/noisy?
- Report at least one representative error per important confusion pair, with true label, predicted label, short text excerpt, and a reasoned interpretation.
- Include hardware, seed, versions, input normalization, vectorizer/vocabulary sizes, neural architecture, optimizer, learning rate, batch size, early-stopping method, and train/test protocol for reproducibility.
- Training time is machine-dependent and does not include download time or necessarily all notebook overhead. Report the exact values produced by your run.

The teammate handling broader data analysis can reuse `class_distribution.csv`, `per_class_test_metrics.csv`, `test_confusion_pairs.csv`, `representative_test_errors.csv`, and `run_summary.json`.

## 20. Optional: zip the complete output directory for download

Run this final cell after all outputs are generated. Kaggle writes the zip under `/kaggle/working/`, which can then be downloaded from the notebook Output panel.

In [ ]:
import shutil
archive_base = OUTPUT_DIR.parent / "ag_news_task1_outputs"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print("Created:", archive_path)
print(f"Size: {Path(archive_path).stat().st_size / (1024 * 1024):.2f} MB")